# Session 3 — Track A: Production RAG Capstone — McKinsey Knowledge Assistant (Student)

In this capstone, you will build a **production-grade McKinsey Knowledge Assistant** from scratch -- a RAG service that answers consulting questions by retrieving from a curated knowledge base of industry reports, strategic frameworks, and engagement playbooks.

You will integrate everything from Days 1-3: embeddings, vector stores, chunking, query transformation, caching, monitoring, and evaluation -- into a single deployable system serving McKinsey consultants and partners.

Work through the milestones in order. Each milestone builds on the previous one. Prepare a **2-minute demo** for Session 4.

## Capstone Objectives

By the end of this capstone, you will have built a **McKinsey Knowledge Assistant** comprising:

1. A **document ingestion pipeline** that indexes McKinsey consulting content (frameworks, playbooks, industry reports) with smart chunking and rich metadata (practice area, industry, engagement type, confidentiality level)
2. A **retrieval engine** with query expansion and reranking, tuned for consulting questions from partners and engagement managers
3. A **generation layer** that produces executive-ready answers with citations, MECE structure, and strategic relevance
4. A **production wrapper** with caching, monitoring, and evaluation measuring consulting quality (actionability, strategic depth, executive readiness)

In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env

# LangSmith tracing configuration
os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
import json
import time
import hashlib
import numpy as np
from datetime import datetime
from collections import defaultdict
from typing import List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.text_splitter import RecursiveCharacterTextSplitter

print("All imports successful!")

---
## Milestone 1: Document Ingestion Pipeline

Build a pipeline that ingests McKinsey consulting knowledge assets -- frameworks, playbooks, industry reports, and CEO briefing templates -- applies smart chunking, embeds them, and indexes them in ChromaDB. This is the foundation of your Knowledge Assistant.

Each document carries rich consulting metadata: practice area (e.g., Operations, M&A, Digital), industry vertical, engagement type, and confidentiality level. This metadata enables filtered retrieval by context.

**Requirements:**
- Accept documents with text, source, and consulting metadata (practice_area, industry, engagement_type, confidentiality)
- Chunk using a configurable strategy (recursive, markdown-aware)
- Embed chunks and store in ChromaDB with all metadata preserved
- Report ingestion statistics (chunk count, avg size, etc.)

In [ ]:
# Milestone 1 - Document Ingestion Pipeline

# TODO: Build a DocumentIngester class that:
# 1. Takes McKinsey consulting documents with metadata (practice_area, industry, engagement_type, confidentiality)
# 2. Chunks them using RecursiveCharacterTextSplitter
# 3. Embeds chunks using OpenAI embeddings
# 4. Stores in ChromaDB with metadata (source, chunk_index, total_chunks, practice_area, industry, etc.)
#
# Hint: Each chunk should inherit metadata from its parent document (practice area, industry, etc.)
# Hint: Track chunk_index so you can reconstruct document order
# Hint: Use collection name "mckinsey_knowledge" for the ChromaDB collection

class DocumentIngester:
    def __init__(self, collection_name="mckinsey_knowledge", chunk_size=int(os.getenv("CHUNK_SIZE", "300")), chunk_overlap=int(os.getenv("CHUNK_OVERLAP", "50"))):
        # YOUR CODE HERE
        pass

    def ingest(self, documents: List[Dict]) -> Dict:
        """Ingest documents. Each doc: {text: str, source: str, metadata: dict}"""
        # YOUR CODE HERE
        pass


# McKinsey consulting knowledge base for testing
# test_docs = [
#     {
#         "text": "McKinsey's post-merger integration (PMI) framework emphasizes three pillars: capturing synergies within the first 100 days, aligning organizational culture through structured leadership workshops, and establishing a clean-room integration management office (IMO). Research across 200+ acquisitions shows that companies capturing 80% of synergies in year one outperform peers by 15% in total shareholder returns.",
#         "source": "pmi_framework_2024.md",
#         "metadata": {"practice_area": "M&A", "industry": "Cross-industry", "engagement_type": "Framework", "confidentiality": "Internal"}
#     },
#     {
#         "text": "Digital transformation in manufacturing requires a phased approach: Phase 1 -- Assess digital maturity across the value chain using McKinsey's Digital Quotient (DQ) diagnostic. Phase 2 -- Prioritize use cases by impact and feasibility (typically predictive maintenance, demand sensing, and quality analytics yield highest ROI). Phase 3 -- Build a scalable data platform and upskill the workforce.",
#         "source": "digital_manufacturing_playbook.md",
#         "metadata": {"practice_area": "Digital", "industry": "Manufacturing", "engagement_type": "Playbook", "confidentiality": "Client-ready"}
#     },
#     {
#         "text": "Operations excellence programs typically deliver 15-25% cost reduction in the first 18 months. McKinsey's approach begins with a diagnostic spanning procurement, production, and logistics. Lean management principles are combined with advanced analytics to identify waste. A critical success factor is embedding a performance management infrastructure -- daily management systems, KPI cascades, and capability-building academies.",
#         "source": "ops_excellence_guide.md",
#         "metadata": {"practice_area": "Operations", "industry": "Cross-industry", "engagement_type": "Implementation Guide", "confidentiality": "Internal"}
#     },
#     {
#         "text": "CEO briefing templates should follow the Situation-Complication-Resolution (SCR) structure. Open with a one-sentence situation framing the strategic context. Introduce the complication -- the specific challenge or opportunity requiring action. Close with the resolution -- McKinsey's recommended path forward with 2-3 actionable next steps. Keep the core narrative to 5-7 pages maximum.",
#         "source": "ceo_briefing_template.md",
#         "metadata": {"practice_area": "Leadership", "industry": "Cross-industry", "engagement_type": "Template", "confidentiality": "Internal"}
#     },
#     {
#         "text": "Supply chain resilience has become a board-level priority following recent global disruptions. McKinsey's supply chain diagnostic evaluates five dimensions: supplier diversification, inventory buffering strategy, logistics network flexibility, digital visibility (control tower maturity), and workforce agility. Best-in-class companies maintain dual sourcing for critical components and hold 4-6 weeks of safety stock for high-risk SKUs.",
#         "source": "supply_chain_resilience_report.md",
#         "metadata": {"practice_area": "Operations", "industry": "Manufacturing", "engagement_type": "Industry Report", "confidentiality": "Client-ready"}
#     }
# ]
# ingester = DocumentIngester()
# stats = ingester.ingest(test_docs)
# print(stats)

---
## Milestone 2: Advanced Retrieval Engine

Build a retrieval engine that handles the types of questions McKinsey consultants and partners ask: strategic queries that may span multiple practice areas, industry-specific questions, and requests for specific frameworks or best practices.

The engine must support query expansion (since consultants phrase the same question in many ways), deduplication, and LLM reranking to surface the most strategically relevant content.

**Requirements:**
- Multi-query expansion (generate 3 alternative consulting phrasings)
- Retrieve and deduplicate across all query variants
- Rerank results using LLM scoring for strategic relevance and actionability
- Return results with relevance scores and consulting metadata

In [ ]:
# Milestone 2 - Advanced Retrieval Engine

# TODO: Build a RetrievalEngine class that:
# 1. Expands consulting queries into multiple variants (different phrasings a partner might use)
# 2. Retrieves from the ChromaDB collection for each variant
# 3. Deduplicates results
# 4. Reranks using LLM-as-judge (score 0-10 for strategic relevance and actionability)
#
# Hint: Use the ingester's collection from Milestone 1
# Hint: When expanding queries, prompt the LLM to think like a McKinsey consultant
# Hint: Reranking prompt should consider strategic relevance, not just keyword match

class RetrievalEngine:
    def __init__(self, collection):
        # YOUR CODE HERE
        pass

    def retrieve(self, query, k=5) -> List[Dict]:
        """Retrieve top-k relevant knowledge base chunks with scores."""
        # YOUR CODE HERE
        pass


# Test with consulting questions
# engine = RetrievalEngine(ingester.collection)
# results = engine.retrieve("What is McKinsey's approach to post-merger integration?")
# for r in results:
#     print(f"{r['score']}/10 [{r['metadata']['source']}] ({r['metadata'].get('practice_area', 'N/A')}) {r['text'][:80]}...")

---
## Milestone 3: Generation Layer with Citations

Build the generation component that produces executive-ready consulting answers. Responses must cite sources (framework documents, playbooks, reports), follow a structured format suitable for partner review, and flag when the knowledge base lacks sufficient information to answer confidently.

**Requirements:**
- Generate answers grounded in retrieved knowledge base context
- Include source citations in the response (e.g., [Source: pmi_framework_2024.md])
- Structure responses for an executive audience -- MECE where applicable, concise and actionable
- Detect and flag when the context is insufficient for a confident answer
- Support configurable system prompts tailored to consulting standards

In [ ]:
# Milestone 3 - Generation Layer with Citations

# TODO: Build a RAGGenerator class that:
# 1. Takes retrieved knowledge base chunks and a consulting question
# 2. Formats context with source tags and practice area for citation
# 3. Generates an executive-ready answer with inline citations
# 4. Flags low-confidence answers when context is insufficient
#
# Hint: Include source info in the context: [Source: filename | Practice: area]
# Hint: System prompt should instruct the LLM to cite sources and use MECE structure
# Hint: System prompt should frame responses for an executive audience
# Hint: Check if the LLM says "does not contain sufficient" or similar -> flag as low confidence

class RAGGenerator:
    def __init__(self, model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")):
        # YOUR CODE HERE
        pass

    def generate(self, question, retrieved_chunks) -> Dict:
        """Generate an executive-ready, cited answer from retrieved chunks."""
        # YOUR CODE HERE
        pass


# Test with a consulting question
# generator = RAGGenerator()
# answer = generator.generate("What is McKinsey's approach to post-merger integration?", retrieved_chunks)
# print(f"Answer: {answer['content']}")
# print(f"Sources: {answer['sources']}")
# print(f"Confidence: {answer['confidence']}")

---
## Milestone 4: Production Wrapper -- Caching, Monitoring & Evaluation

Wrap everything in a production McKinsey Knowledge Assistant with caching, monitoring, cost tracking, and quality evaluation tailored to consulting standards.

Evaluation metrics go beyond generic RAG quality -- measure **strategic relevance** (does the answer address the consulting question?), **faithfulness** (is it grounded in the knowledge base?), and **executive readiness** (is it structured, actionable, and suitable for partner/client review?).

**Requirements:**
- Semantic caching to avoid redundant LLM calls for repeated partner questions
- Request logging with latency, tokens, and cost
- Automated evaluation (strategic_relevance, faithfulness, executive_readiness)
- A dashboard method that reports Knowledge Assistant health metrics

In [ ]:
# Milestone 4 - Production McKinsey Knowledge Assistant

# TODO: Build a ProductionRAGService class that combines:
# 1. DocumentIngester (from Milestone 1) -- for McKinsey knowledge base
# 2. RetrievalEngine (from Milestone 2) -- for consulting query retrieval
# 3. RAGGenerator (from Milestone 3) -- for executive-ready answers
# 4. Caching, monitoring, and consulting-quality evaluation
#
# Hint: Pipeline: check cache -> retrieve -> generate -> evaluate -> cache & log
# Hint: Use LLM-as-judge for evaluation with consulting-specific metrics:
#       - strategic_relevance: Does it address the consulting need?
#       - faithfulness: Is it grounded in the knowledge base?
#       - executive_readiness: Is it structured and actionable for partner review?
# Hint: Dashboard should show: total queries, cache hit rate, avg latency, avg consulting quality

class ProductionRAGService:
    def __init__(self, documents):
        # YOUR CODE HERE
        pass

    def query(self, question) -> Dict:
        """Full RAG pipeline with caching, monitoring, evaluation."""
        # YOUR CODE HERE
        pass

    def dashboard(self) -> Dict:
        """Return McKinsey Knowledge Assistant health metrics."""
        # YOUR CODE HERE
        pass


# Test with realistic consulting questions
# service = ProductionRAGService(test_docs)
#
# questions = [
#     "What is McKinsey's approach to post-merger integration?",
#     "Summarize best practices for supply chain digitization in manufacturing",
#     "What is McKinsey's approach to post-merger integration?",  # cached
#     "How should I structure a CEO briefing for an operations transformation?"
# ]
#
# for q in questions:
#     result = service.query(q)
#     cached_str = "CACHED" if result["cached"] else f"quality={result['metrics']['overall']}"
#     print(f"[{cached_str:12s}] {q[:55]:55s} | {result['latency_ms']:.0f}ms")
#     print(f"  Answer: {result['answer'][:120]}...\n")
#
# print("--- McKinsey Knowledge Assistant Dashboard ---")
# for k, v in service.dashboard().items():
#     print(f"  {k}: {v}")

---
## Capstone Summary

You have built a **production-grade McKinsey Knowledge Assistant** with:

1. **Ingestion** -- Smart chunking of consulting assets (frameworks, playbooks, industry reports) with rich metadata (practice area, industry, engagement type, confidentiality level)
2. **Retrieval** -- Multi-query expansion and LLM reranking tuned for consulting questions from partners and engagement managers
3. **Generation** -- Executive-ready answers with source citations, MECE structure, and confidence flagging
4. **Production** -- Caching, monitoring, and automated evaluation measuring strategic relevance, faithfulness, and executive readiness

Prepare a **2-minute demo** for Session 4. Focus on what makes your McKinsey Knowledge Assistant production-ready: how it handles diverse consulting queries, evaluates answer quality, and serves partners reliably.

**Up Next:** Session 4 -- Cross-track presentations, governance, and closing.